# Coverage of the UCI predictive distributions

Every run gives posterior draws $(\theta_s, \sigma_s)$, $s = 1, \dots, S$. At a test input $x_i$ the posterior predictive is the equally weighted Gaussian mixture

$$F_i(y) = \frac{1}{S} \sum_{s=1}^{S} \Phi\left(\frac{y - f_{\theta_s}(x_i)}{\sigma_s}\right),$$

which is the same predictive the NLL and CRPS in the UCI tables use. The noise enters through $\Phi$, so the intervals are for $y$ and not only for $f(x)$.

**PIT value.** $u_i = F_i(y_i)$, the predictive CDF at the observed response. A calibrated predictive gives $u_i \sim \mathrm{Uniform}(0, 1)$.

**Coverage.** The central interval at level $p$ runs from $F_i^{-1}\big(\tfrac{1-p}{2}\big)$ to $F_i^{-1}\big(\tfrac{1+p}{2}\big)$. Since $F_i$ is increasing, $y_i$ lies in it exactly when $|u_i - \tfrac12| \le \tfrac p2$, so

$$C(p) = \frac{1}{N} \sum_{i=1}^{N} \mathbf{1}\big\{|u_i - \tfrac12| \le \tfrac p2\big\}.$$

All levels come from the same $u_i$, with no interval computed explicitly. Calibration means $C(p) = p$ for every $p$. $C(p) > p$ means intervals that are too wide, $C(p) < p$ too narrow.

**Width.** $W(p)$ is the mean interval length on the original $y$ scale. The quantiles $F_i^{-1}$ are found by bisection. Coverage alone rewards wide intervals, so compare $W(p)$ between models with similar $C(p)$.

**Calibration error.** $\mathrm{CE} = \frac{1}{99} \sum_{p} |C(p) - p|$ over $p = 0.01, 0.02, \dots, 0.99$. One number per run, $0$ is perfect.

**Noise in the estimates.** Under calibration $N\,C(p) \sim \mathrm{Bin}(N, p)$, so the standard error is $\sqrt{p(1-p)/N}$. Boston has $N = 51$ test points per split, which gives about $0.04$ at $p = 0.9$. The tables average over splits and the figure pools the $u_i$ of all splits.

**Notes.**
- Coverage belongs to the model and the posterior approximation together. An exact posterior of a misspecified model need not reach $C(p) = p$.
- LBBNN stores one fixed $\sigma$ for all draws, and it is used as stored, as in the NLL and CRPS.
- The rows follow the paper tables (`PAPER_ROWS` in `uci_results_utils.py`). Sticky rows are labelled with their prior inclusion weight $w$.


In [ ]:
from pathlib import Path
import os
if Path.cwd().name == "notebooks":
    os.chdir("..")

import matplotlib.pyplot as plt
from IPython.display import display

from notebooks.utils import coverage_utils as cu

SAVE_DIR = Path("results/plots_v2/coverage")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
ARCHS    = ("small", "medium")
DATASETS = ("boston", "concrete", "energy", "yacht")

# first call computes every run, later calls read results/paper_v2/_coverage_cache.pkl
recs = cu.collect(ARCHS, DATASETS)
df = cu.per_split_frame(recs)

## Overview

Calibration error CE for every model and dataset, mean over splits. Lower is better.

In [ ]:
for arch in ARCHS:
    print(f"=== {arch}, CE")
    display(cu.overview(df, arch, "CE"))

In [ ]:
# the same view for coverage and width at one level
for arch in ARCHS:
    for col in ("C(0.9)", "W(0.9)"):
        print(f"=== {arch}, {col}")
        display(cu.overview(df, arch, col))

In [ ]:
import numpy as np
print(cu.overview(df, "medium", "C(0.9)")['boston'])


## Per dataset

Mean ± standard error over splits for every level.

In [ ]:
for arch in ARCHS:
    for dataset in DATASETS:
        print(f"=== {arch}, {dataset}")
        display(cu.summary_table(df, arch, dataset))

## Calibration curves

$C(p) - p$ against $p$, with the $u_i$ of all splits pooled. Above zero means too wide, below zero too narrow. The grey band is $\pm 2$ binomial standard errors for the pooled test set. Colour gives the sampler family, solid lines are sticky with $w = 0.3$, dotted sticky with $w = 0.1$, dashed the non sparse models.

In [ ]:
# ROWS = None shows every model, or pick some, e.g. ["NUTS", "Sticky ZigZag (w=0.3)", "LBBNN"]
ROWS = None

for arch in ARCHS:
    fig = cu.calibration_figure(recs, arch, DATASETS, rows=ROWS)
    fig.suptitle(arch, y=1.18)
    plt.show()
    fig.savefig(SAVE_DIR / f"calibration_{arch}.png", bbox_inches="tight", dpi=200)